In [3]:
import numpy as np
from pathlib import Path
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()

from copy import deepcopy

# Progress bar
from tqdm.auto import tqdm
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torchvision import datasets, transforms

import tensorboard as tb
from torch.utils.tensorboard import SummaryWriter

In [ ]:
class RNN(nn.module):

    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch_size, seq_len, input_size)

        output, hidden = self.rnn(x)

        # hidden: (num_layers, batch_size, hidden_size)
        last_hidden = hidden[-1]   # pak laatste laag

        logits = self.fc(last_hidden)
        return logits

In [4]:
TRAIN_DATA = 'train_data.csv'
df = pd.read_csv(TRAIN_DATA)
print(df.head())

           id                                         FalseSent  \
0  sentence_1                                I sting a mosquito   
1  sentence_2                            A giraffe is a person.   
2  sentence_3  A normal closet is larger than a walk-in closet.   
3  sentence_4                       I like to ride my chocolate   
4  sentence_5                    A GIRL WON THE RACE WITH HORSE   

                                             OptionA  \
0                                A human is a mammal   
1              Giraffes can drink water from a lake.   
2                Walk-in closets are normal closets.   
3           Chocolate is delicious and bikes are not   
4  GIRL HAVE BEAUTIFUL HAIR BUT THE HORSE DOESN'T...   

                                            OptionB  \
0                             A human is omnivorous   
1                   A giraffe is not a human being.   
2           A person can sleep in a walk-in closet.   
3    Chocolate is a food, not a transpor